# 03c — Anomalías (condiciones extremas)

Ejecuta el §6.2 de `planning/plan_unsupervised_implementation.md` sobre los conjuntos A y B de 03a.
Tres detectores, tasa por percentil (nunca a ojo), bandera por consenso (≥2 de 3) y cuatro contrastes para separar extremos físicos de artefactos. Visualización con el PCA existente (sin dependencias nuevas).

## 03c.0 — Imports (celda única del cuaderno)

Como siempre: todo lo importable, aquí. Detectores y banderas del módulo (con tests); en el cuaderno solo se orquestan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from wildfire.data.io import load_csv, save_csv
from wildfire.processing.pca_weather import pca_input_columns
from wildfire.processing.unsupervised import (
    ID_COL,
    ANOMALY_PCT, CONSENSUS_MIN_VOTES,
    check_ids, consensus_flag, encode_franja_horaria, fit_isolation_forest, fit_lof,
    imputed_blh_mask, percentile_flag, resolve_set, robust_mahalanobis,
    weather_pc_scores,
)

## 03c.1 — Carga, escalado y referencias de contraste

Se cargan `features_a/b.csv` y el perfilado, se codifica la hora en B y se estandariza. Además se marcan las filas con BLH exactamente en la mediana (577,5: probablemente imputadas, no medidas) — la BLH se lee del CSV base porque `features_a` ya trae PCs en vez de las variables crudas — y se fija el umbral del 15 % superior de FRP, la referencia del mapa 5.3 de 02b para el contraste de severidad.


In [ ]:
sets_raw = {
    name: load_csv(f"data/processed/unsupervised/features_{name}.csv")
    for name in ("a", "b")
}
perfilado = load_csv("data/processed/unsupervised/perfilado.csv")
ids = check_ids(sets_raw["a"], sets_raw["b"], perfilado)
frp = perfilado["frp"].to_numpy()
sets_raw["b"] = encode_franja_horaria(sets_raw["b"])
X = {name: pd.DataFrame(StandardScaler().fit_transform(frame.drop(columns=[ID_COL])), columns=[c for c in frame.columns if c != ID_COL], index=frame.index)
     for name, frame in sets_raw.items()}
blh_imputada = imputed_blh_mask(load_csv("data/processed/expanded/firms_spain_final_lags_index.csv"))
umbral_frp15 = float(pd.Series(frp).quantile(0.85))
print(f"Filas: {len(perfilado):,} | BLH imputada: {int(blh_imputada.sum()):,} | FRP top-15% desde {umbral_frp15:.1f}")


## 03c.2 — Tres detectores (convención: más alto = más raro)

Cada fila se puntúa con tres detectores: Isolation Forest (aislamiento global), LOF (rareza local: raro respecto a sus vecinos) y Mahalanobis robusta (lejanía elíptica global con correlaciones). Nota histórica: con las 14 meteorológicas crudas, `MinCovDet` ahogaba la salida con >2.000 `RuntimeWarning` por ejecución; desde que A/B son PCs (decorrelacionados por construcción) no se emite ninguno, así que el filtro `catch_warnings` de la celda siguiente queda comentado —visible, no global— y se restauraría si reaparecieran.

Por qué avisaba antes (cadena causal, sin ecuaciones): `MinCovDet` busca la mitad de los datos con la covarianza más compacta (algoritmo FastMCD), y en cada paso de refinamiento el determinante debería encoger — sklearn avisa cuando un paso lo agranda. Había dos detonantes: (a) variables meteorológicas casi colineales (temperatura/humedad/VPD/ET0 se mueven juntas por física — el PC1 ya lo mostró), que aplanan la covarianza y la vuelven nerviosa — eliminado por construcción con los PCs; (b) el pico de filas con BLH exactamente en la mediana de imputación (nuestra propia imputación), una masa puntual que rompe el supuesto de distribución elíptica — diluido al repartirse la BLH entre los 7 PCs. Consecuencia práctica (entonces y ahora): el ajuste puede quedar ligeramente subóptimo, pero el *ranking* (qué filas quedan más lejos) sobrevive — y todo lo que sigue es por ranking (percentiles, consenso), nunca por distancias crudas.

Cómo leer cada figura:
- Histogramas (fila superior, un panel por conjunto A/B): eje x = score de rareza (más alto = más raro, por convención), eje y = nº de filas; tres colores = tres detectores. Se lee la cola derecha, no el bulto: una colita separada = un puñado estructuralmente distinto; colas solapadas entre colores = acuerdo sobre *quién* es extremo.
- Cajas (fila inferior): caja = masa normal (mediana + IQR); puntos sueltos tras los bigotes = la cantera de extremos que ve cada detector. Si los puntos de un detector empequeñecen a los demás, ese método dispara más fácil — el consenso existe para arbitrar eso.
- Tabla de descriptivos: medias/desviaciones/mín/máx por detector, como ancla de cordura (p. ej. distancias Mahalanobis de miles reflejan la escala χ² en 7–12 dimensiones, no un error).

In [ ]:
scores = {}
for _name, _X in X.items():
    # Antes se silenciaban aquí los RuntimeWarning de MinCovDet (import warnings +
    # catch_warnings): los provocaban las 14 meteorológicas crudas, casi colineales.
    # Con los PCs decorrelacionados ya no hace falta; si reaparecieran, se restauraría.
    _scores = pd.DataFrame({
        "isolation_forest_score": fit_isolation_forest(_X),
        "lof_score": fit_lof(_X),
        "mahalanobis_score": robust_mahalanobis(_X),
    }, index=_X.index)
    scores[_name] = _scores
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for _ax, (_sname, _s) in zip(axes[0], scores.items()):
    for _c in _s.columns:
        _ax.hist(_s[_c].to_numpy(), bins=60, alpha=0.6, label=_c)
    _ax.legend(fontsize=8)
    _ax.set_title(f"Scores ({_sname}): cola derecha = raro")
for _ax, (_sname, _s) in zip(axes[1], scores.items()):
    _ax.boxplot([_s[_c].dropna().to_numpy() for _c in _s.columns], tick_labels=list(_s.columns))
    _ax.set_title(f"Cajas ({_sname})")
plt.tight_layout()
plt.show()
display(scores["a"].describe().T[["mean", "std", "min", "max"]])


*Tres colas derechas presentes; Mahalanobis en escala χ² (std ~5000) frente a IForest/LOF (~0,04/0,6) — por eso se comparan percentiles, no valores.*

## 03c.3 — Banderas por percentil + consenso, y guardado

Se marca el 2 % superior de cada score (`ANOMALY_PCT`, plan §6.2) y se conserva lo que marquen al menos dos detectores. Queda guardado en `anomalias_{a,b}.csv` (scores con sufijo `_score`, banderas con sufijo `_flag` y `consenso_flag`, índice común — sin nombres duplicados entre scores y banderas).

Cómo leer la tabla de acuerdo: cada celda = % de filas que marcan ambos detectores a la vez. Acuerdo alto entre dos métodos = ven lo mismo; acuerdo bajo en los tres pares = cada uno caza rarezas distintas y el consenso es conservador (pocas filas, pero sólidas).


In [ ]:
banderas = {}
for _name, _s in scores.items():
    _flags = pd.DataFrame(
        {f"{c.removesuffix('_score')}_flag": percentile_flag(_s[c]) for c in _s.columns},
        index=_s.index)
    _flags["consenso_flag"] = consensus_flag(*[_flags[c] for c in _flags.columns])
    banderas[_name] = _flags
    _out = pd.concat([_s, _flags], axis=1)
    _out.insert(0, ID_COL, ids)
    save_csv(_out, f"data/processed/unsupervised/anomalias_{_name}.csv")
    print(f"{_name}: consenso {int(_flags['consenso_flag'].sum()):,} filas ({100.0 * _flags['consenso_flag'].mean():.2f} %)")
_det = [c.removesuffix("_score") for c in scores["a"].columns]
display(pd.DataFrame(
    [[100 * float((banderas["a"][f"{_i}_flag"] & banderas["a"][f"{_j}_flag"]).mean()) for _j in _det] for _i in _det],
    index=_det, columns=_det).round(1))
print("(% de filas marcadas por ambos detectores a la vez, conjunto A)")


*Consenso A: 249 filas (1,08 %); B: 171 (0,74 %). IForest∩Mahalanobis 1,0 %; LOF casi disjunto (0,1 %): caza rarezas distintas.*

## 03c.4 — Cuatro contrastes (físico frente a artefacto)

La bandera de consenso se cruza contra cuatro referencias: severidad (top-15 % FRP), mezcla CCAA×mes, filas BLH-imputadas y regímenes KMeans de 03b (k=5 ilustrativo).

Cómo leer la tabla: cada fila es un contraste con su veredicto operativo. 'Físico' = la rareza viene acompañada de virulencia o de una ola concreta (VPD alto + viento + suelo seco a la vez). 'Artefacto' = la rareza es un pico aislado de BLH justo en la mediana imputada o una racha del sensor sin virulencia.

In [ ]:
_f = banderas["a"]["consenso_flag"].to_numpy()
_top_frp = frp >= umbral_frp15
_rows = [
    ("Solape con top-15% FRP", f"{100.0 * (_f & _top_frp).sum() / max(_f.sum(), 1):.1f} % de anómalas son muy virulentas"),
    ("Anómalas no virulentas", f"{int((_f & ~_top_frp).sum()):,} anómalas fuera del top-FRP (condiciones raras sin fuego fiero)"),
    ("BLH imputada entre anómalas", f"{100.0 * (_f & blh_imputada.to_numpy()).sum() / max(_f.sum(), 1):.1f} % (artefacto candidato si alto)"),
    ("CCAA top anómala", str(perfilado.loc[_f, "ccaa"].value_counts().head(3).to_dict())),
    ("Mes top anómalo", str(perfilado.loc[_f, "month"].value_counts().head(3).to_dict())),
]
contrastes = pd.DataFrame(_rows, columns=["contraste", "lectura"])
display(contrastes)
save_csv(contrastes, "data/processed/unsupervised/contrastes_anomalias.csv")
_km5_df = load_csv("data/processed/unsupervised/clustering_kmeans_a.csv")
check_ids(_km5_df, perfilado)
_km5 = _km5_df["k_5"].to_numpy()
print(pd.crosstab(pd.Series(_km5[_f], name="grupo_k5"), columns="anómalas").to_string())
print("(reparto de las anómalas entre los 5 regímenes de 03b: ¿vienen de un régimen o de todos?)")

## 03c.5 — Visualización en el PCA existente (sin dependencias nuevas)

Las filas se proyectan al plano PC1–PC2 del PCA meteorológico (el mismo 91,5 % de 02b) y se pintan el consenso y el FRP encima, más el mapa geográfico de anómalas.

Cómo leer las figuras:
- PC1–PC2 con consenso: si las anómalas forman una nube separada en un extremo (p. ej. PC1 alto = sequedad/calor extremas), son un régimen extremo real; si salpican por todas partes, son rarezas heterogéneas sin firma común.
- PC1–PC2 con FRP: compara a ojo si virulencia y rareza viven en el mismo rincón (deberían solaparse parcialmente según 03b.5, no totalmente: hay fuegos fieros en condiciones normales).
- Mapa: anómalas concentradas en una zona/fecha = episodio concreto (ola de calor); repartidas = propiedad estructural de las colas.

In [ ]:
_pca_in = load_csv("data/processed/expanded/firms_spain_final_lags_index.csv")
check_ids(_pca_in, perfilado)
_pcs = weather_pc_scores(_pca_in)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
_sc = axes[0].scatter(_pcs["PC1"], _pcs["PC2"], c=banderas["a"]["consenso_flag"], s=3, cmap="coolwarm")
axes[0].set_title("Consenso anómalo sobre PC1–PC2")
axes[1].scatter(_pcs["PC1"], _pcs["PC2"], c=np.log1p(frp), s=3, cmap="hot")
axes[1].set_title("log1p(FRP) sobre PC1–PC2")
_coords = _pca_in[["longitude", "latitude"]]
axes[2].scatter(_coords["longitude"], _coords["latitude"], c=banderas["a"]["consenso_flag"], s=3, cmap="coolwarm")
axes[2].set_title("Consenso anómalo (mapa)")
plt.show()
print("03c medición completa: 03e decide si score_anomalia promociona.")